# Nettoyage du corpus ISO

Pipeline reproductible de nettoyage du corpus extrait du PDF ISO 7240-14.

**Problème** : l'extraction PDF produit trois types de bruit :
1. Caracteres entrelaces dans les mots (`Panardt s1e4r:vice` au lieu de `Part 14: service`)
2. Lignes non-textuelles (copyright IHS, sommaire, legendes de figures, formules)
3. Lignes en francais melangees a l'anglais

**Methode** (suivant la recommandation : ne pas corriger a la main, utiliser un LLM) :
- Etape 1 : pre-filtre par regles (retire les cas evidents -> moins d'appels LLM)
- Etape 2 : le LLM traite le reste par petits paquets, repare ce qui est recuperable,
  supprime l'irrecuperable, recolle les phrases et ne garde que l'anglais.


In [ ]:
!pip install -q anthropic

In [ ]:
import os, re
from anthropic import Anthropic

# Cle API : Runtime > ... ou via les secrets Colab.
# Mets ta cle dans la variable d'environnement ANTHROPIC_API_KEY
client = Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))
MODEL = "claude-sonnet-4-5"   # adapte au modele disponible

RAW_PATH   = "scenarios_fire_detection_and_alarm_systems.txt"  # corpus brut (upload)
CLEAN_PATH = "scenarios_fire_detection_clean.txt"              # sortie

In [ ]:
from google.colab import files
up = files.upload()                      # uploader le .txt brut
RAW_PATH = list(up.keys())[0]
print("Corpus brut :", RAW_PATH)

## Etape 1 — Pre-filtre par regles

On retire les lignes manifestement inexploitables avant d'appeler le LLM :
copyright/IHS, sommaire (`....39`), lignes trop courtes, charabia dense.
Ca reduit le nombre d'appels LLM (donc le cout et le temps).

In [ ]:
with open(RAW_PATH, encoding="utf-8") as f:
    lines = [l.strip() for l in f if l.strip()]

NOISE = re.compile(r"(copyright|all rights reserved|reproduction|IHS|Not for Resale|"
                   r"Licensee|ISO/IEC Directives|iso\.org|geneva|case postale|"
                   r"e-mail|web www|Provided by|networking permitted|MST)", re.I)

def is_toc(l):
    return bool(re.search(r"\.{4,}\s*\d+\s*$", l))

def garble(l):
    weird = len(re.findall(r"[a-z][A-Z]|[A-Za-z]\d|\d[A-Za-z]|\(cid:|[©•]", l))
    return weird / max(len(l.split()), 1)

prefiltered = []
for l in lines:
    if len(l) < 15:          continue
    if NOISE.search(l):      continue
    if is_toc(l):            continue
    if garble(l) > 1.2:      continue   # seuil large : le LLM gere le reste
    prefiltered.append(l)

print(f"Brut : {len(lines)} lignes -> apres pre-filtre : {len(prefiltered)} lignes")

## Etape 2 — Nettoyage par LLM (par paquets)

Le LLM recoit des paquets de lignes et applique une consigne stricte :
reparer le charabia recuperable, supprimer l'irrecuperable, ne garder que
l'anglais, recoller les fragments en phrases completes.

In [ ]:
SYSTEM = """You clean noisy text extracted from a PDF of the ISO 7240-14 fire-detection standard.
Rules:
- Keep ONLY English. Drop any French line.
- Repair lines where characters are scrambled by PDF extraction, ONLY if the intended meaning is clear.
- DELETE (output nothing for) lines that are copyright notices, tables of contents, figure/table captions, page headers/footers, or formulas/symbols too corrupted to recover.
- Merge fragments that belong to the same sentence into one complete sentence.
- Output one clean sentence per line. No numbering, no commentary, no markdown."""

def clean_batch(batch_lines):
    text = "\n".join(batch_lines)
    msg = client.messages.create(
        model=MODEL, max_tokens=2000,
        system=SYSTEM,
        messages=[{"role": "user", "content": f"Clean these lines:\n\n{text}"}],
    )
    out = "".join(b.text for b in msg.content if b.type == "text")
    return [l.strip() for l in out.splitlines() if l.strip()]

BATCH = 25
cleaned = []
for i in range(0, len(prefiltered), BATCH):
    batch = prefiltered[i:i+BATCH]
    cleaned.extend(clean_batch(batch))
    print(f"  paquet {i//BATCH + 1}/{(len(prefiltered)-1)//BATCH + 1} traite", end="\r")

print(f"\nNettoyage termine : {len(cleaned)} phrases propres")

In [ ]:
# Deduplication + sauvegarde
seen, final = set(), []
for l in cleaned:
    k = l.lower()
    if k not in seen:
        seen.add(k); final.append(l)

with open(CLEAN_PATH, "w", encoding="utf-8") as f:
    f.write("\n".join(final))

print(f"Corpus propre : {len(final)} phrases -> {CLEAN_PATH}")
print("\nApercu :")
for l in final[:8]:
    print("  +", l)

In [ ]:
from google.colab import files
files.download(CLEAN_PATH)